# <b>Librerias y Dependencias</b>

In [ ]:
import ODM

# <b> Creacion de Clases </b>

In [2]:
ODM.initApp()

# <b> Ejercicios </b>

## *Ejercicio 1*
Listado de todas personas que han estudiado en la UPM o UAM.

In [ ]:
cursor_1 = ODM.Persona.aggregate([
    # Descomponemos el array de estudios para contabilizar los estudios de forma individual
    {"$unwind": "$estudios"},
    # Buscamos las personas que contengan dentro de su campo estudios, en su campo universidad las universidades que se piden
    {"$match":
        {"estudios.universidad":{"$in":["UPM","UAM"]}}
    },
    # Project sirve para renombrar para añadir una columna (p.Ej: numPersonas) o, cómo en este caso; seleccionar cuales se muestran y cuales no
    {"$project": {
        "_id": 0,       
        "nombre": 1,
        "dni": 1,
        "estudios":1
    }}
])

In [49]:
for doc in cursor_1:
    print(doc)

{'nombre': 'Raul Fernandez', 'dni': '67890123F', 'estudios': {'universidad': 'UAM', 'inicio': '2016-09-01', 'fin': '2018-06-30'}}
{'nombre': 'Ana Navarro', 'dni': '90123456I', 'estudios': {'universidad': 'UPM', 'inicio': '2011-09-01', 'fin': '2013-06-30'}}
{'nombre': 'Carlos Perez', 'dni': '23456789B', 'estudios': {'universidad': 'UPM', 'inicio': '2010-09-01', 'fin': '2014-06-30'}}
{'nombre': 'Laura Gomez', 'dni': '12345678A', 'estudios': {'universidad': 'UPM', 'inicio': '2017-09-01', 'fin': '2019-06-30'}}


## *Ejercicio 2*
Listado de las diferentes universidades en las que han estudiado las personas residentes en Madrid.

In [ ]:
cursor_2 = ODM.Persona.aggregate([
    # Buscamos las personas en las que su direccion contengan la palabra Madrid
    {"$match": {"direccion": {"$regex": "Madrid", "$options":"i"}}},    # $regex se utiliza para las expresiones regular $option:i para que no sea case-sensitive; en este caso, Madrid
    # Descoponemos el array de estudios para contabilizar los estudios de forma individual
    {"$unwind": "$estudios"},
    # agrupamos las personas por las universidades que hayan estudiado y por cada persona que haya estudiado en esa universidad incrementamos su num_students
    {"$group":
        {"_id": "$estudios.universidad", "num_students": {"$sum":1}}
    },
    # en este caso va a mostrar las columnas universidad y num_students, en universidad el id del Group y num_students su valor por cada grupo 
    {
        "$project": {
            "_id": 0,
            "universidad": "$_id",
            "num_students": 1
        }
    }
])

In [47]:
for doc in cursor_2:
    print(doc)

{'num_students': 1, 'universidad': 'Deusto'}
{'num_students': 1, 'universidad': 'UPM'}
{'num_students': 1, 'universidad': 'UAM'}
{'num_students': 2, 'universidad': 'UCM'}


## *Ejercicio 3*
Personas que, en la descripción de su perfil, incluye los términos “Big Data” o “Inteligencia Artificial”

In [ ]:
cursor_3 = ODM.Persona.aggregate([
    # Buscamos las personas en las que su descripción contengan la palabra Big Data o Inteligencia artificial
    {"$match": {"descripcion": {"$regex":"(Big Data|Inteligencia Artificial)","$options":"i"}}},
    # Mostramos solo el nombre, el dni y la descripcion de dichas personas
    {"$project": {
        "_id": 0,       
        "nombre": 1,
        "dni": 1,
        "descripcion": 1
    }}
])

In [44]:
for doc in cursor_3:
    print(doc)

{'nombre': 'Ana Navarro', 'dni': '90123456I', 'descripcion': 'Profesora universitaria en inteligencia artificial.'}
{'nombre': 'Maria Lopez', 'dni': '34567890C', 'descripcion': 'Analista de datos con experiencia en machine learning y big data.'}
{'nombre': 'Laura Gomez', 'dni': '12345678A', 'descripcion': 'Ingeniera de software especializada en inteligencia artificial.'}


## *Ejercicio 4*
Guarda en una tabla nueva el listado de las personas que ha terminado alguno de sus estudios en el 2017 o después.

In [ ]:
cursor_4 = ODM.Persona.aggregate([
    # Descomponemos el array de estudios para contabilizar los estudios de forma individual
    {"$unwind":"$estudios"},
    {"$match": {"$expr": {"$gte": [{"$year": {"$toDate": "$estudios.fin"}},2017]}}}, # $expr permite usar expresiones de agregación, usamos $toDate porque el dato esta guardado como cadena
    # $out crea una nueva coleccion con el resultado del agregate y con el nombre especificado
    {"$out":"Grads_Post_2017"}
])

![image.png](ej4_result.png)

## *Ejercicio 5*
Calcular el número medio de estudios realizados por las personas que han trabajado o trabajan en la Microsoft.

In [ ]:
cursor_5 = ODM.Persona.aggregate([
    # Descomponemos el array de trabajos para contabilizar los trabajos de forma individual
    {"$unwind":"$trabajos"},
    # Buscamos las personas en las que su trabajo, en el campo nombre, contengan Microsoft
    {"$match": {"trabajos.empresa": "Microsoft"}},
    # Crea un nuevo campo num_estudios por cada persona que contiene la longitud del array estudios
    {"$project": {"num_estudios":{"$size":"$estudios"}}},
    # Agrupa estas personas por las empresas en las que trabaje y calcula la media de estudios en esta agrupacion
    # obteniendo la media de estudios de los trabajadores (actuales o posteriores) de Microsoft
    {"$group":
        {
            "_id":"$trabajos.empresa",
            "avg_estudios":{"$avg":"$num_estudios"}
        }
    },
    {"$project":{
        "_id":0,
        "avg_estudios":1
    }}
])

In [42]:
for doc in cursor_5:
    print(doc)

{'avg_estudios': 1.2}


## *Ejercicio 6*
Distancia media al trabajo (distancia geodésica) de los actuales trabajadores de Google. Se pueden indicar las coordenadas de la oficina de Google
manualmente.

In [18]:
coord_google = {
    "type": "Point",
    "coordinates": [-3.6883, 40.4531]  # [longitud, latitud]
}

In [ ]:
cursor_6 = ODM.Persona.aggregate([
    {"$geoNear":{
        "near":coord_google, # fijamos las coordenadas de google
        "distanceField": "distancia_m", # el resultado será la columna distancia_m
        "spherical":True    # la tierra es esférica 
    }},
    # Descoponemos el array de trabajos para contabilizar los trabajos de forma individual
    {"$unwind":"$trabajos"},
    # Seleccionamos las personas cuyo trabajo sea Google y sin fecha de "fin", (en los datos se guarda con null); ya que queremos los actuales
    {"$match":{"trabajos.empresa":"Google","trabajos.fin":None}},
    # Agrupamos todos los resultados para poder calcular la distancia media entre todos
    {"$group":{
        "_id":None,
        "avg_distance":{"$avg":{"$divide":["$distancia_m",1000]}}   #   dividimos para expresarlo en km
    }},
    {"$project":{
        "_id":0,
        "avg_distance":1
    }}
])

In [40]:
for doc in cursor_6:
    print(doc)

{'avg_distance': 249.5500468362552}


## *Ejercicio 7*
Listado de las tres universidades que más veces aparece como centro de estudios de las personas registradas. 
Mostrar universidad y el número de veces que aparece.

In [ ]:
cursor_7 = ODM.Persona.aggregate([
    # Descoponemos el array de estudios para contabilizar los estudios de forma individual
    {"$unwind":"$estudios"},
    # agrupamos las personas por las universidades que hayan estudiado y por cada persona que haya estudiado en esa universidad incrementamos su num_students
    {
        "$group": {
            "_id": "$estudios.universidad",
            "numPersonas": { "$sum": 1 }
        }
    },
    # ordenamos de mayor a menor
    {"$sort": {"numPersonas": -1}},
    # mostramos solo 3
    {"$limit": 3},
    {
        "$project": {
            "_id": 0,
            "universidad": "$_id",
            "numPersonas": 1
        }
    }
])

In [33]:
for doc in cursor_7:
    print(doc)

{'numPersonas': 4, 'universidad': 'UCM'}
{'numPersonas': 3, 'universidad': 'UPM'}
{'numPersonas': 1, 'universidad': 'UPV'}
